# 2. Feature Engineering I


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

from ift6758.features.simple_visualizations import games_list, distance


In [ ]:
# to acquire raw play-by-play from 2016/17 to 2023/24, go to the file "raw_data_acquisition.ipynb" and run the code in there

# create training / test set (2023/24)
train_years = [2016, 2017, 2018, 2019, 2020, 2021, 2022]
test_years = [2023]

# getting the games for the training ds
train_tidy = []
for year in train_years:
    train_tidy.append(games_list(year))

train_tidy = pd.concat(train_tidy)
train_tidy.head()

In [ ]:
# saving the tidy dataset to pickle
train_tidy.to_pickle("train_tidy_basic.pkl")

In [ ]:
# Verifying the dataset has the necessary columns:
print(train_tidy.head())
print(train_tidy.columns)
print(train_tidy[train_tidy["emptyNet"]==True])

In [ ]:
# Histogram of shot/goal according to distance
dist_df = distance(train_tidy)

plt.hist(dist_df[dist_df["event_type"]=="shot-on-goal"]["distance"], 16, label="Shot on Goal")
plt.hist(dist_df[dist_df["event_type"]=="goal"]["distance"], 16, label="Goal")

plt.legend()
plt.xlabel("Distance")
plt.ylabel("Number of shots")
plt.title(f"Distance Distribution of Shots and Goals for {train_years[0]}-{train_years[-1]}")
plt.show()

In [ ]:
def angles(dataframe) -> pd.DataFrame:
    """
    Modified version of "distance" function from Milestone 1
    Function to add the columns distance and angle to the dataframe.
    Distance is the distance between the goal and the coordinates where the shot was taken
    Angle is the angle of the shot relative to the goal

    Parameters:
        dataframe (pd.DataFrame) : Dataframe of the games

    Returns:
        pd.DataFrame
    """
    df = dataframe.copy()
    # team_defending: True is home_team, False is away_team
    # df["team_id"] contain "eventOwnerTeamId" which is the team associated with the action
    # df["team_id"] is the team that is trying to score
    df["team_defending"] = (df["team_id"] != df["homeTeam_id"])

    df["side_defending"] = df["homeTeamDefendingSide"]

    conditions = [df["side_defending"]=="right",df["side_defending"]=="left"]
    choices = ["left","right"]

    df["side_defending"] = np.where(df["team_defending"]==False,np.select(conditions,choices),df["side_defending"])

    # if right, goal_coor = (89,0)
    # else left, (-89,0)
    df["goalCoordX"] = np.where(df["side_defending"]=="right",89,-89)

    #calculate the distance between the x,y coordinates and the goal
    #the y coordinate for the goals is 0

    #the sign depends of the side during the period e.g. goalX = -57 for left side, 57 for right side. Better of with absolute value
    dx = df["goalCoordX"]-df["x_coord"]


    df["distance"] = np.sqrt((dx)**2 + df["y_coord"]**2)

    #df["angle"] = np.arctan(df["y_coord"]/(df["goalCoordX"]-df["x_coord"]))
    df["angle"] = np.arctan(df["y_coord"]/(dx))

    #Remove team_defending, side_defending, goalCoordX
    #Keep only the distance
    df = df.drop(
        columns = ["team_defending", "side_defending", "goalCoordX"]
    )

    return df

In [ ]:
# Histogram for the angles of shots/goals
angles_df = angles(train_tidy)
print(angles_df.head())

#plt.hist(angles_df["angle"], 16, label="Shot on Goal")

plt.hist(angles_df[angles_df["event_type"]=="shot-on-goal"]["angle"], label="Shot on Goal")
plt.hist(angles_df[angles_df["event_type"]=="goal"]["angle"], label="Goal")

plt.legend()
plt.xlabel("Angle (radians)")
plt.ylabel("Number of shots")
plt.title(f"Angle Distribution of Shots and Goals for {train_years[0]}-{train_years[-1]}")
plt.show()

In [ ]:
# 2d histogram of angles and distances

#print(angles_df.head())
#print(angles_df[angles_df["distance"].isna()].head())

# Remove missing coords:
angles_df.replace(np.inf, np.nan, inplace=True)
angles_df.dropna(subset=["distance", "angle"], inplace=True)
sns.jointplot(x="angle", y="distance", data=angles_df,  hue="event_type", kind="hist", marginal_kws=dict(bins=25))

plt.title(f"Distance and angle of shot and goals for {train_years[0]}-{train_years[-1]}")
plt.plot()

In [ ]:
# saving the angles dataset to pickle, n'inclut pas les distances nan ou inf
angles_df.to_pickle("train_basic.pkl")

In [ ]:
# Figure showing the goals scored from far away when the net is empty vs not

goals = dist_df[dist_df["event_type"]=="goal"]

plt.hist([goals[goals["emptyNet"]==True]["distance"], goals[goals["emptyNet"]==False]["distance"]], 25, histtype="bar", label=["Empty Net", "Not Empty"])
#plt.hist(, 25, label="Not Empty", histtype="bar")

# To see only the goals that are from very far:
plt.xlim(50)
plt.ylim(0,500)

plt.legend()
plt.xlabel("Distance")
plt.ylabel("Number of goals")
plt.title(f"Distance Distribution of Goals for {train_years[0]}-{train_years[-1]}")

plt.plot()

event 657 of 2016020428: no coordinates